Cell 1：基础路径

In [1]:
from pathlib import Path
import os
import sys
import json
import shutil
import subprocess

REPO_URL = "https://github.com/huqian122/SpaMGCL.git"

CLONE_DIR = Path("/kaggle/working/SpaMGCL_repo")
PROJECT_DIR = CLONE_DIR / "SpaMGCL"

DATA_ROOT = Path("/kaggle/input/datasets/wuvdji/smgc-data")

print("Python:", sys.version)
print("DATA_ROOT:", DATA_ROOT)
print("Data exists:", DATA_ROOT.exists())

assert DATA_ROOT.exists(), f"找不到数据目录: {DATA_ROOT}"

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
DATA_ROOT: /kaggle/input/datasets/wuvdji/smgc-data
Data exists: True


Cell 2：先检查 E18.5 数据是否真的存在

In [2]:
E185_DIR = DATA_ROOT / "E18.5_mouse_brain"

RNA_FILE = E185_DIR / "adata_RNA.h5ad"
ATAC_FILE = E185_DIR / "adata_ATAC.h5ad"

print("E18.5 dir :", E185_DIR)
print("RNA      :", RNA_FILE, RNA_FILE.exists())
print("ATAC     :", ATAC_FILE, ATAC_FILE.exists())

assert RNA_FILE.exists(), RNA_FILE
assert ATAC_FILE.exists(), ATAC_FILE

print("\nE18.5 DATA CHECK: PASS")

E18.5 dir : /kaggle/input/datasets/wuvdji/smgc-data/E18.5_mouse_brain
RNA      : /kaggle/input/datasets/wuvdji/smgc-data/E18.5_mouse_brain/adata_RNA.h5ad True
ATAC     : /kaggle/input/datasets/wuvdji/smgc-data/E18.5_mouse_brain/adata_ATAC.h5ad True

E18.5 DATA CHECK: PASS


Cell 3：从 GitHub 拉最新代码

In [3]:
if CLONE_DIR.exists():
    shutil.rmtree(CLONE_DIR)

subprocess.run(
    [
        "git",
        "clone",
        "--depth",
        "1",
        REPO_URL,
        str(CLONE_DIR),
    ],
    check=True,
)

assert PROJECT_DIR.exists(), (
    f"项目目录不存在: {PROJECT_DIR}\n"
    f"请检查 GitHub 仓库结构。"
)

print("Clone directory :", CLONE_DIR)
print("Project directory:", PROJECT_DIR)

Cloning into '/kaggle/working/SpaMGCL_repo'...


Clone directory : /kaggle/working/SpaMGCL_repo
Project directory: /kaggle/working/SpaMGCL_repo/SpaMGCL


Cell 4：记录当前 commit

In [4]:
COMMIT = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=CLONE_DIR,
    text=True,
).strip()

BRANCH = subprocess.check_output(
    ["git", "branch", "--show-current"],
    cwd=CLONE_DIR,
    text=True,
).strip()

print("Branch:", BRANCH)
print("Commit:", COMMIT)

Branch: main
Commit: 4343c48d75f5cffab9222fa41a62585fe7f0dd00


Cell 5：检查 clean 文件是否都在 GitHub 版本里

In [5]:
required_files = [
    "experiments/run_exp.py",
    "src/clustering/predict.py",
    "src/data/dataset.py",
    "scripts/audit_run.py",
    "configs/final_clean/e185_clean_smoke.yaml",
    "configs/final_clean/e185_clean_200.yaml",
    "configs/final_clean/hlna1_clean_200.yaml",
    "configs/final_clean/d1_clean_200.yaml",
    "configs/final_clean/s2e15_clean_200.yaml",
    "configs/final_clean/s2e18_clean_200.yaml",
]

for rel in required_files:
    path = PROJECT_DIR / rel
    print(f"{rel:<55} {path.exists()}")
    assert path.exists(), f"缺少文件: {path}"

print("\nCLEAN FILE CHECK: PASS")

experiments/run_exp.py                                  True
src/clustering/predict.py                               True
src/data/dataset.py                                     True
scripts/audit_run.py                                    True
configs/final_clean/e185_clean_smoke.yaml               True
configs/final_clean/e185_clean_200.yaml                 True
configs/final_clean/hlna1_clean_200.yaml                True
configs/final_clean/d1_clean_200.yaml                   True
configs/final_clean/s2e15_clean_200.yaml                True
configs/final_clean/s2e18_clean_200.yaml                True

CLEAN FILE CHECK: PASS


Cell 6：依赖检查

In [6]:
# 固定 SpaMGCL 需要的科学计算环境，避免 anndata 自动升级 numpy 导致 scipy/sklearn 冲突

import sys
import subprocess

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-cache-dir",
        "numpy==2.0.2",
        "scipy==1.14.1",
        "scikit-learn==1.5.2",
        "pandas==2.2.3",
        "h5py==3.12.1",
        "anndata==0.11.4",
        "zarr==2.18.7",
    ],
    check=True,
)

print("PACKAGE INSTALL: PASS")
print("安装完成后，请重启 Kaggle Session，然后从 Cell 1 重新运行。")

PACKAGE INSTALL: PASS
安装完成后，请重启 Kaggle Session，然后从 Cell 1 重新运行。


In [7]:
import numpy as np
import scipy
import sklearn
import pandas as pd
import anndata
import h5py
import torch

print("numpy   :", np.__version__)
print("scipy   :", scipy.__version__)
print("sklearn :", sklearn.__version__)
print("pandas  :", pd.__version__)
print("anndata :", anndata.__version__)
print("h5py    :", h5py.__version__)
print("torch   :", torch.__version__)

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

print("\nCORE IMPORT CHECK: PASS")

numpy   : 2.0.2
scipy   : 1.14.1
sklearn : 1.5.2
pandas  : 2.2.3
anndata : 0.11.4
h5py    : 3.12.1
torch   : 2.10.0+cu128

CORE IMPORT CHECK: PASS


Cell 7：先检查 E18.5 的两个 h5ad

In [8]:
import anndata as ad

rna = ad.read_h5ad(RNA_FILE, backed="r")
atac = ad.read_h5ad(ATAC_FILE, backed="r")

print("RNA shape :", rna.shape)
print("ATAC shape:", atac.shape)
print("RNA spots :", len(rna.obs_names))
print("ATAC spots:", len(atac.obs_names))

assert len(rna.obs_names) == len(atac.obs_names)

rna.file.close()
atac.file.close()

print("\nH5AD READ CHECK: PASS")

RNA shape : (2129, 32285)
ATAC shape: (2129, 161461)
RNA spots : 2129
ATAC spots: 2129

H5AD READ CHECK: PASS


Cell 8：检查 GPU

In [9]:
import torch

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
else:
    print("WARNING: 当前没有 GPU")

Torch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
CUDA version: 12.8


Cell 9：静态编译检查

In [10]:
subprocess.run(
    [
        sys.executable,
        "-m",
        "py_compile",
        "experiments/run_exp.py",
        "src/clustering/predict.py",
        "src/data/dataset.py",
        "scripts/audit_run.py",
    ],
    cwd=PROJECT_DIR,
    check=True,
)

print("PYTHON COMPILE: PASS")

PYTHON COMPILE: PASS


Cell 10：读取并严格检查 E18.5 smoke 配置

In [11]:
import yaml

CONFIG_PATH = PROJECT_DIR / "configs/final_clean/e185_clean_smoke.yaml"

with CONFIG_PATH.open("r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

assert cfg["experiment"]["dataset"] == "E18.5"
assert int(cfg["experiment"]["seed"]) == 0
assert int(cfg["training"]["epochs"]) == 50
assert int(cfg["training"]["warm_up_epochs"]) == 10

assert int(cfg["model"]["fine_dim"]) == 128
assert float(cfg["model"]["tau_s"]) == 0.3
assert float(cfg["model"]["cluster_temperature"]) == 0.3

assert float(cfg["loss"]["lambda_rec"]) == 1.0
assert float(cfg["loss"]["lambda_mgcl"]) == 3.0
assert float(cfg["loss"]["lambda_cluster"]) == 0.1
assert float(cfg["loss"]["lambda_spatial"]) == 0.0

assert cfg["spatial"]["enabled"] is False
assert cfg["clustering"]["method"] == "kmeans"
assert cfg["clustering"]["embedding"] == "concat_z"
assert int(cfg["clustering"]["n_clusters"]) == 14

print("E18.5 CLEAN SMOKE CONFIG: PASS")

E18.5 CLEAN SMOKE CONFIG: PASS


Cell 11：启动 50 轮 smoke：

In [12]:
cmd = [
    sys.executable,
    "experiments/run_exp.py",
    "--config",
    "configs/final_clean/e185_clean_smoke.yaml",
]

subprocess.run(
    cmd,
    cwd=PROJECT_DIR,
    check=True,
)

Dataset: E18.5 | spots=2129 | device=cuda
Modalities: RNA, ATAC | label=Combined_Clusters
Spatial graph: shape=(2129, 2129), nnz=7784
Learning rates: backbone=0.001 | cluster_head=0.008 (8x)
epoch 001/050 | total=23.166977 | rec=0.160814 | mgcl=7.668721 | cluster=3.298035 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=8.967e-03 | neg_count=4530512 | snf_masked_positions=0 | effC=13.976 | gradC=0.000e+00
epoch 002/050 | total=23.126160 | rec=0.148333 | mgcl=7.659276 | cluster=3.297573 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=7.602e-03 | neg_count=4530512 | snf_masked_positions=0 | effC=13.982 | gradC=0.000e+00
epoch 003/050 | total=23.084681 | rec=0.137188 | mgcl=7.649164 | cluster=3.297174 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=7.515e-03 | neg_count=4530512 | snf_masked_positions=0 | effC=13.986 | gradC=0.000e+00
ep

CompletedProcess(args=['/usr/bin/python3', 'experiments/run_exp.py', '--config', 'configs/final_clean/e185_clean_smoke.yaml'], returncode=0)

Cell 12：检查 E18.5 smoke 输出文件是否完整

In [13]:
RESULT_DIR = PROJECT_DIR / "results_clean/e185_clean_smoke"

required_outputs = [
    "metrics.json",
    "config.yaml",
    "manifest.json",
    "pred_labels.npy",
    "pred_concat_z_kmeans.npy",
    "pred_q_argmax.npy",
    "gt_labels.npy",
    "z_concat.npy",
    "z_mean.npy",
]

print("Result directory:")
print(RESULT_DIR)
print()

for name in required_outputs:
    path = RESULT_DIR / name
    print(f"{name:<35} {path.exists()}")
    assert path.exists(), f"缺少结果文件: {path}"

print("\nRESULT ARTIFACT CHECK: PASS")

Result directory:
/kaggle/working/SpaMGCL_repo/SpaMGCL/results_clean/e185_clean_smoke

metrics.json                        True
config.yaml                         True
manifest.json                       True
pred_labels.npy                     True
pred_concat_z_kmeans.npy            True
pred_q_argmax.npy                   True
gt_labels.npy                       True
z_concat.npy                        True
z_mean.npy                          True

RESULT ARTIFACT CHECK: PASS


Cell 13：运行正式实验审计 audit_run.py

In [14]:
subprocess.run(
    [
        sys.executable,
        "scripts/audit_run.py",
        str(RESULT_DIR),
    ],
    cwd=PROJECT_DIR,
    check=True,
)

Dataset: E18.5
Seed: 0
Cluster warm-up epochs: 10
Cluster loss start epoch: 11
Configured lambda_cluster: 0.1
Official embedding: concat_z
Official clustering method: kmeans
Epochs: 50
lambda_rec: 1.0
lambda_mgcl: 3.0
lambda_cluster: 0.1
lambda_spatial: 0.0
ARI: 0.4418224877017169
NMI: 0.5324422753770279
NMI average method: max
recomputed ARI: 0.4418224877017169
recomputed NMI: 0.5324422753770279
Q argmax diagnostic ARI: 0.49644983313595575
Q argmax diagnostic NMI: 0.4893469082517356
z_concat shape: (2129, 512)
PASS


CompletedProcess(args=['/usr/bin/python3', 'scripts/audit_run.py', '/kaggle/working/SpaMGCL_repo/SpaMGCL/results_clean/e185_clean_smoke'], returncode=0)

Cell 14：独立重新计算正式 ARI、NMI 并验证保存标签

In [15]:
import json
import numpy as np

from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
)

with (RESULT_DIR / "metrics.json").open("r", encoding="utf-8") as f:
    metrics = json.load(f)

gt = np.load(RESULT_DIR / "gt_labels.npy")
pred = np.load(RESULT_DIR / "pred_labels.npy")
pred_concat = np.load(RESULT_DIR / "pred_concat_z_kmeans.npy")
pred_q = np.load(RESULT_DIR / "pred_q_argmax.npy")
z_concat = np.load(RESULT_DIR / "z_concat.npy")

# 1. 正式标签必须就是 concat-Z + KMeans
assert np.array_equal(pred, pred_concat)

# 2. embedding 不允许出现 NaN / Inf
assert np.isfinite(z_concat).all()

# 3. 独立重新计算正式指标
nmi_method = metrics["nmi_average_method"]

ari = adjusted_rand_score(gt, pred)

nmi = normalized_mutual_info_score(
    gt,
    pred,
    average_method=nmi_method,
)

# 4. Q 仅作为 diagnostic
q_ari = adjusted_rand_score(gt, pred_q)

q_nmi = normalized_mutual_info_score(
    gt,
    pred_q,
    average_method=nmi_method,
)

print("Official concat-Z + KMeans")
print("--------------------------")
print("ARI =", ari)
print("NMI =", nmi)

print()
print("Q argmax diagnostic")
print("-------------------")
print("ARI =", q_ari)
print("NMI =", q_nmi)

print()
print("z_concat shape =", z_concat.shape)

# 5. 必须与 metrics.json 顶层结果一致
assert abs(float(metrics["ARI"]) - ari) < 1e-12
assert abs(float(metrics["NMI"]) - nmi) < 1e-12

print("\nINDEPENDENT METRIC CHECK: PASS")

Official concat-Z + KMeans
--------------------------
ARI = 0.4418224877017169
NMI = 0.5324422753770279

Q argmax diagnostic
-------------------
ARI = 0.49644983313595575
NMI = 0.4893469082517356

z_concat shape = (2129, 512)

INDEPENDENT METRIC CHECK: PASS


Cell 15：独立检查 cluster warm-up 是否严格按照 1–10 轮关闭

In [16]:
history = metrics["loss_history"]

assert len(history) == 50, f"Expected 50 epochs, got {len(history)}"

for row in history:
    epoch = int(row["epoch"])
    effective_cluster = float(row["effective_lambda_cluster"])

    # 前 10 轮关闭 cluster loss
    if epoch <= 10:
        expected_cluster = 0.0
    else:
        expected_cluster = 0.1

    assert abs(effective_cluster - expected_cluster) < 1e-12, (
        f"Epoch {epoch}: "
        f"effective_lambda_cluster={effective_cluster}, "
        f"expected={expected_cluster}"
    )

    # reconstruction 和 sample contrastive 不受 warm-up 影响
    assert abs(float(row["lambda_rec"]) - 1.0) < 1e-12
    assert abs(float(row["lambda_mgcl"]) - 3.0) < 1e-12

print("Epoch 1-10  : cluster loss OFF")
print("Epoch 11-50 : lambda_cluster = 0.1")
print()
print("CLUSTER WARM-UP CHECK: PASS")

Epoch 1-10  : cluster loss OFF
Epoch 11-50 : lambda_cluster = 0.1

CLUSTER WARM-UP CHECK: PASS


Cell 16：汇总 E18.5 clean smoke 最终结果

In [18]:
print("=" * 72)
print("E18.5 CLEAN SMOKE SUMMARY")
print("=" * 72)

print("Git branch :", BRANCH)
print("Git commit :", COMMIT)

print()
print("Dataset :", metrics["dataset"])
print("Seed    :", metrics["seed"])
print("Epochs  :", metrics["epochs"])

print()
print("Official readout")
print("----------------")
print("Embedding :", metrics["clustering_embedding"])
print("Method    :", metrics["clustering_method_used"])

print()
print("Configured loss")
print("----------------")
print(metrics["configured_loss"])

print()
print("Warm-up")
print("-------")
print("Warm-up epochs      :", metrics["cluster_warm_up_epochs"])
print("Cluster start epoch :", metrics["cluster_loss_start_epoch"])

print()
print("OFFICIAL RESULT: concat-Z + KMeans")
print("----------------------------------")
print(f"ARI = {metrics['ARI']:.6f}")
print(f"NMI = {metrics['NMI']:.6f}")

print()
print("Q ARGMAX DIAGNOSTIC")
print("-------------------")
print(f"ARI = {q_ari:.6f}")
print(f"NMI = {q_nmi:.6f}")

print()
print("Representation")
print("--------------")
print("z_concat shape =", z_concat.shape)

print()
print("Final WD weights")
print("----------------")
print(metrics.get("final_weights"))

print("=" * 72)

E18.5 CLEAN SMOKE SUMMARY
Git branch : main
Git commit : 4343c48d75f5cffab9222fa41a62585fe7f0dd00

Dataset : E18.5
Seed    : 0
Epochs  : 50

Official readout
----------------
Embedding : concat_z
Method    : kmeans

Configured loss
----------------
{'lambda_rec': 1.0, 'lambda_mgcl': 3.0, 'lambda_cluster': 0.1, 'lambda_spatial': 0.0}

Warm-up
-------
Warm-up epochs      : 10
Cluster start epoch : 11

OFFICIAL RESULT: concat-Z + KMeans
----------------------------------
ARI = 0.441822
NMI = 0.532442

Q ARGMAX DIAGNOSTIC
-------------------
ARI = 0.496450
NMI = 0.489347

Representation
--------------
z_concat shape = (2129, 512)

Final WD weights
----------------
[0.2501639127731323, 0.2495821714401245, 0.24989604949951172, 0.25035783648490906]


Cell 17：检查 E18.5 200 轮正式配置是否与 smoke 仅相差训练轮数

In [19]:
import yaml
from copy import deepcopy

SMOKE_CONFIG_PATH = PROJECT_DIR / "configs/final_clean/e185_clean_smoke.yaml"
FINAL_CONFIG_PATH = PROJECT_DIR / "configs/final_clean/e185_clean_200.yaml"

with SMOKE_CONFIG_PATH.open("r", encoding="utf-8") as f:
    smoke_cfg = yaml.safe_load(f)

with FINAL_CONFIG_PATH.open("r", encoding="utf-8") as f:
    final_cfg = yaml.safe_load(f)

print("Smoke epochs:", smoke_cfg["training"]["epochs"])
print("Final epochs:", final_cfg["training"]["epochs"])

assert int(smoke_cfg["training"]["epochs"]) == 50
assert int(final_cfg["training"]["epochs"]) == 200

assert int(smoke_cfg["experiment"]["epochs"]) == 50
assert int(final_cfg["experiment"]["epochs"]) == 200

# 复制后移除允许不同的字段
smoke_compare = deepcopy(smoke_cfg)
final_compare = deepcopy(final_cfg)

smoke_compare["experiment"]["name"] = "__SAME__"
final_compare["experiment"]["name"] = "__SAME__"

smoke_compare["experiment"]["epochs"] = "__SAME__"
final_compare["experiment"]["epochs"] = "__SAME__"

smoke_compare["training"]["epochs"] = "__SAME__"
final_compare["training"]["epochs"] = "__SAME__"

assert smoke_compare == final_compare, (
    "e185_clean_smoke.yaml 与 e185_clean_200.yaml "
    "除了 experiment.name 和 epochs 之外还有其他差异！"
)

print()
print("Smoke and 200-epoch configs are identical except:")
print("- experiment.name")
print("- experiment.epochs")
print("- training.epochs")
print()
print("E18.5 200 CONFIG CONSISTENCY CHECK: PASS")

Smoke epochs: 50
Final epochs: 200

Smoke and 200-epoch configs are identical except:
- experiment.name
- experiment.epochs
- training.epochs

E18.5 200 CONFIG CONSISTENCY CHECK: PASS


Cell 18：确认没有旧的 E18.5 200 轮结果

In [20]:
FINAL_RESULT_DIR = PROJECT_DIR / "results_clean/e185_clean_200"

print("Final result directory:")
print(FINAL_RESULT_DIR)

if (FINAL_RESULT_DIR / "metrics.json").exists():
    raise RuntimeError(
        "发现旧的 e185_clean_200/metrics.json。\n"
        "为了避免复用旧结果，请不要继续运行。"
    )

print("\nE18.5 200 OUTPUT CLEAN CHECK: PASS")

Final result directory:
/kaggle/working/SpaMGCL_repo/SpaMGCL/results_clean/e185_clean_200

E18.5 200 OUTPUT CLEAN CHECK: PASS


Cell 19：正式运行 E18.5 clean 200 epochs

In [21]:
cmd = [
    sys.executable,
    "experiments/run_exp.py",
    "--config",
    "configs/final_clean/e185_clean_200.yaml",
]

print("Running E18.5 official 200-epoch experiment")
print("Command:")
print(" ".join(cmd))
print()

subprocess.run(
    cmd,
    cwd=PROJECT_DIR,
    check=True,
)

print("\nE18.5 CLEAN 200 TRAINING FINISHED")

Running E18.5 official 200-epoch experiment
Command:
/usr/bin/python3 experiments/run_exp.py --config configs/final_clean/e185_clean_200.yaml

Dataset: E18.5 | spots=2129 | device=cuda
Modalities: RNA, ATAC | label=Combined_Clusters
Spatial graph: shape=(2129, 2129), nnz=7784
Learning rates: backbone=0.001 | cluster_head=0.008 (8x)
epoch 001/200 | total=23.166977 | rec=0.160814 | mgcl=7.668721 | cluster=3.298035 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=8.967e-03 | neg_count=4530512 | snf_masked_positions=0 | effC=13.976 | gradC=0.000e+00
epoch 002/200 | total=23.126160 | rec=0.148333 | mgcl=7.659276 | cluster=3.297573 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=7.602e-03 | neg_count=4530512 | snf_masked_positions=0 | effC=13.982 | gradC=0.000e+00
epoch 003/200 | total=23.084681 | rec=0.137188 | mgcl=7.649164 | cluster=3.297174 | spatial_loss=0.000000 | lambda_spatial=0 | s

Cell 20：审计 E18.5 200 轮正式实验结果

In [22]:
FINAL_RESULT_DIR = (
    PROJECT_DIR / "results_clean/e185_clean_200"
)

subprocess.run(
    [
        sys.executable,
        "scripts/audit_run.py",
        str(FINAL_RESULT_DIR),
    ],
    cwd=PROJECT_DIR,
    check=True,
)

Dataset: E18.5
Seed: 0
Cluster warm-up epochs: 10
Cluster loss start epoch: 11
Configured lambda_cluster: 0.1
Official embedding: concat_z
Official clustering method: kmeans
Epochs: 200
lambda_rec: 1.0
lambda_mgcl: 3.0
lambda_cluster: 0.1
lambda_spatial: 0.0
ARI: 0.3288632452496322
NMI: 0.5339714780957494
NMI average method: max
recomputed ARI: 0.3288632452496322
recomputed NMI: 0.5339714780957494
Q argmax diagnostic ARI: 0.33174717342016236
Q argmax diagnostic NMI: 0.5094275032714655
z_concat shape: (2129, 512)
PASS


CompletedProcess(args=['/usr/bin/python3', 'scripts/audit_run.py', '/kaggle/working/SpaMGCL_repo/SpaMGCL/results_clean/e185_clean_200'], returncode=0)

Cell 21：打包保存 E18.5 200 轮完整实验结果

In [23]:
import shutil
from pathlib import Path

ZIP_BASE = Path(
    "/kaggle/working/e185_clean_200"
)

zip_path = shutil.make_archive(
    str(ZIP_BASE),
    "zip",
    root_dir=FINAL_RESULT_DIR,
)

print("E18.5 200 result package:")
print(zip_path)

E18.5 200 result package:
/kaggle/working/e185_clean_200.zip


Cell 24：检查 200轮实验内部 epoch50、epoch100、epoch200 的聚类变化

In [24]:
import json

FINAL_DIR = PROJECT_DIR / "results_clean/e185_clean_200"

rows = []

for epoch in [50, 100, 200]:
    path = FINAL_DIR / f"metrics_epoch{epoch}.json"

    with path.open("r", encoding="utf-8") as f:
        m = json.load(f)

    rows.append(
        (
            epoch,
            float(m["ARI"]),
            float(m["NMI"]),
        )
    )

print("=" * 60)
print("E18.5 CHECKPOINT CLUSTERING PERFORMANCE")
print("=" * 60)

for epoch, ari, nmi in rows:
    print(
        f"Epoch {epoch:>3d} | "
        f"ARI = {ari:.6f} | "
        f"NMI = {nmi:.6f}"
    )

print("=" * 60)

E18.5 CHECKPOINT CLUSTERING PERFORMANCE
Epoch  50 | ARI = 0.421887 | NMI = 0.518872
Epoch 100 | ARI = 0.324594 | NMI = 0.541733
Epoch 200 | ARI = 0.328863 | NMI = 0.533971


Cell 25：对比独立50轮 smoke 与 200轮实验内部 epoch50

In [25]:
import json

SMOKE_DIR = PROJECT_DIR / "results_clean/e185_clean_smoke"
FINAL_DIR = PROJECT_DIR / "results_clean/e185_clean_200"

with (SMOKE_DIR / "metrics.json").open("r", encoding="utf-8") as f:
    smoke = json.load(f)

with (FINAL_DIR / "metrics_epoch50.json").open("r", encoding="utf-8") as f:
    ep50 = json.load(f)

print("Independent 50-epoch smoke")
print("--------------------------")
print(f"ARI = {float(smoke['ARI']):.12f}")
print(f"NMI = {float(smoke['NMI']):.12f}")

print()
print("Epoch-50 checkpoint inside 200-run")
print("----------------------------------")
print(f"ARI = {float(ep50['ARI']):.12f}")
print(f"NMI = {float(ep50['NMI']):.12f}")

print()
print("Difference")
print("----------")
print(
    f"ARI difference = "
    f"{float(ep50['ARI']) - float(smoke['ARI']):+.12f}"
)
print(
    f"NMI difference = "
    f"{float(ep50['NMI']) - float(smoke['NMI']):+.12f}"
)

Independent 50-epoch smoke
--------------------------
ARI = 0.441822487702
NMI = 0.532442275377

Epoch-50 checkpoint inside 200-run
----------------------------------
ARI = 0.421887263854
NMI = 0.518872280776

Difference
----------
ARI difference = -0.019935223847
NMI difference = -0.013569994601


Cell 26：检查独立 50 轮实验内部 epoch50 与最终结果是否完全一致

In [26]:
import json

SMOKE_DIR = PROJECT_DIR / "results_clean/e185_clean_smoke"

with (SMOKE_DIR / "metrics_epoch50.json").open("r", encoding="utf-8") as f:
    smoke_ep50 = json.load(f)

with (SMOKE_DIR / "metrics.json").open("r", encoding="utf-8") as f:
    smoke_final = json.load(f)

ep50_ari = float(smoke_ep50["ARI"])
ep50_nmi = float(smoke_ep50["NMI"])

final_ari = float(smoke_final["ARI"])
final_nmi = float(smoke_final["NMI"])

print("=" * 70)
print("INDEPENDENT 50-EPOCH RUN: INTERNAL CONSISTENCY")
print("=" * 70)

print()
print("metrics_epoch50.json")
print("--------------------")
print(f"ARI = {ep50_ari:.12f}")
print(f"NMI = {ep50_nmi:.12f}")

print()
print("metrics.json")
print("------------")
print(f"ARI = {final_ari:.12f}")
print(f"NMI = {final_nmi:.12f}")

print()
print("Difference: final - epoch50")
print("---------------------------")
print(f"ARI difference = {final_ari - ep50_ari:+.12e}")
print(f"NMI difference = {final_nmi - ep50_nmi:+.12e}")

print("=" * 70)

INDEPENDENT 50-EPOCH RUN: INTERNAL CONSISTENCY

metrics_epoch50.json
--------------------
ARI = 0.441822487702
NMI = 0.532442275377

metrics.json
------------
ARI = 0.441822487702
NMI = 0.532442275377

Difference: final - epoch50
---------------------------
ARI difference = +0.000000000000e+00
NMI difference = +0.000000000000e+00


Cell 27：逐轮比较独立 50 轮与 200 轮实验前 50 轮训练轨迹

In [27]:
import json
import numpy as np

SMOKE_DIR = PROJECT_DIR / "results_clean/e185_clean_smoke"
FINAL_DIR = PROJECT_DIR / "results_clean/e185_clean_200"

with (SMOKE_DIR / "metrics.json").open("r", encoding="utf-8") as f:
    smoke_metrics = json.load(f)

with (FINAL_DIR / "metrics.json").open("r", encoding="utf-8") as f:
    run200_metrics = json.load(f)

h50 = smoke_metrics["loss_history"]
h200 = run200_metrics["loss_history"][:50]

assert len(h50) == 50
assert len(h200) == 50

fields = [
    "total",
    "reconstruction",
    "sample_contrastive",
    "cluster_contrastive",
    "cluster_effective_clusters",
    "cluster_head_gradient_norm",
]

print("=" * 90)
print("50-EPOCH RUN vs FIRST 50 EPOCHS OF 200-EPOCH RUN")
print("=" * 90)

for field in fields:
    a = np.array([float(row[field]) for row in h50])
    b = np.array([float(row[field]) for row in h200])

    diff = np.abs(a - b)

    print()
    print(field)
    print("-" * len(field))

    print(f"epoch 1 diff  = {diff[0]:.12e}")
    print(f"epoch 10 diff = {diff[9]:.12e}")
    print(f"epoch 11 diff = {diff[10]:.12e}")
    print(f"epoch 50 diff = {diff[49]:.12e}")
    print(f"max abs diff  = {diff.max():.12e}")

    idx = np.where(diff > 1e-10)[0]

    if len(idx) == 0:
        print("first diff > 1e-10: NONE")
    else:
        first = int(idx[0])
        print(
            "first diff > 1e-10: "
            f"epoch {first + 1}, "
            f"50run={a[first]:.12f}, "
            f"200run={b[first]:.12f}"
        )

print()
print("=" * 90)

50-EPOCH RUN vs FIRST 50 EPOCHS OF 200-EPOCH RUN

total
-----
epoch 1 diff  = 0.000000000000e+00
epoch 10 diff = 2.670288085938e-05
epoch 11 diff = 9.536743164062e-06
epoch 50 diff = 4.825592041016e-04
max abs diff  = 6.523132324219e-04
first diff > 1e-10: epoch 5, 50run=22.928146362305, 200run=22.928144454956

reconstruction
--------------
epoch 1 diff  = 0.000000000000e+00
epoch 10 diff = 3.725290298462e-08
epoch 11 diff = 5.215406417847e-08
epoch 50 diff = 1.104176044464e-05
max abs diff  = 1.104176044464e-05
first diff > 1e-10: epoch 7, 50run=0.107030421495, 200run=0.107030406594

sample_contrastive
------------------
epoch 1 diff  = 0.000000000000e+00
epoch 10 diff = 9.059906005859e-06
epoch 11 diff = 3.337860107422e-06
epoch 50 diff = 1.659393310547e-04
max abs diff  = 2.112388610840e-04
first diff > 1e-10: epoch 5, 50run=7.602897167206, 200run=7.602896213531

cluster_contrastive
-------------------
epoch 1 diff  = 0.000000000000e+00
epoch 10 diff = 0.000000000000e+00
epoch 11 di

Cell 28：创建第二次 E18.5 50轮重复实验配置

In [28]:
import yaml
from copy import deepcopy

SOURCE_CONFIG = PROJECT_DIR / "configs/final_clean/e185_clean_smoke.yaml"

with SOURCE_CONFIG.open("r", encoding="utf-8") as f:
    repeat_cfg = yaml.safe_load(f)

# 只修改实验名称，其他所有参数保持完全一致
repeat_cfg = deepcopy(repeat_cfg)
repeat_cfg["experiment"]["name"] = "e185_clean_smoke_repeat"

REPEAT_CONFIG = Path("/kaggle/working/e185_clean_smoke_repeat.yaml")

with REPEAT_CONFIG.open("w", encoding="utf-8") as f:
    yaml.safe_dump(
        repeat_cfg,
        f,
        sort_keys=False,
        allow_unicode=True,
    )

print("Repeat config:")
print(REPEAT_CONFIG)

print()
print("Dataset :", repeat_cfg["experiment"]["dataset"])
print("Seed    :", repeat_cfg["experiment"]["seed"])
print("Epochs  :", repeat_cfg["training"]["epochs"])
print("Name    :", repeat_cfg["experiment"]["name"])

assert repeat_cfg["experiment"]["seed"] == 0
assert repeat_cfg["training"]["epochs"] == 50
assert repeat_cfg["clustering"]["embedding"] == "concat_z"
assert repeat_cfg["clustering"]["method"] == "kmeans"

print("\nREPEAT CONFIG CHECK: PASS")

Repeat config:
/kaggle/working/e185_clean_smoke_repeat.yaml

Dataset : E18.5
Seed    : 0
Epochs  : 50
Name    : e185_clean_smoke_repeat

REPEAT CONFIG CHECK: PASS


Cell 29：运行第二次完全相同的 E18.5 50轮实验

In [29]:
REPEAT_RESULT_DIR = (
    PROJECT_DIR / "results_clean/e185_clean_smoke_repeat"
)

if (REPEAT_RESULT_DIR / "metrics.json").exists():
    raise RuntimeError(
        "发现旧的 repeat 结果，请不要覆盖。"
    )

cmd = [
    sys.executable,
    "experiments/run_exp.py",
    "--config",
    str(REPEAT_CONFIG),
]

print("Running repeat experiment...")
print(" ".join(cmd))
print()

subprocess.run(
    cmd,
    cwd=PROJECT_DIR,
    check=True,
)

print("\nE18.5 50-EPOCH REPEAT FINISHED")

Running repeat experiment...
/usr/bin/python3 experiments/run_exp.py --config /kaggle/working/e185_clean_smoke_repeat.yaml

Dataset: E18.5 | spots=2129 | device=cuda
Modalities: RNA, ATAC | label=Combined_Clusters
Spatial graph: shape=(2129, 2129), nnz=7784
Learning rates: backbone=0.001 | cluster_head=0.008 (8x)
epoch 001/050 | total=23.166977 | rec=0.160814 | mgcl=7.668721 | cluster=3.298035 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=8.967e-03 | neg_count=4530512 | snf_masked_positions=0 | effC=13.976 | gradC=0.000e+00
epoch 002/050 | total=23.126160 | rec=0.148333 | mgcl=7.659276 | cluster=3.297573 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=7.602e-03 | neg_count=4530512 | snf_masked_positions=0 | effC=13.982 | gradC=0.000e+00
epoch 003/050 | total=23.084679 | rec=0.137188 | mgcl=7.649164 | cluster=3.297174 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | s

Cell 30：比较两次相同seed的 E18.5 50轮实验

In [32]:
import json
import numpy as np

RUN1_DIR = PROJECT_DIR / "results_clean/e185_clean_smoke"
RUN2_DIR = PROJECT_DIR / "results_clean/e185_clean_smoke_repeat"

with (RUN1_DIR / "metrics.json").open("r", encoding="utf-8") as f:
    m1 = json.load(f)

with (RUN2_DIR / "metrics.json").open("r", encoding="utf-8") as f:
    m2 = json.load(f)

h1 = m1["loss_history"]
h2 = m2["loss_history"]

assert len(h1) == 50
assert len(h2) == 50

print("=" * 72)
print("SAME-SEED 50-EPOCH REPRODUCIBILITY CHECK")
print("=" * 72)

print()
print("Run 1")
print(f"ARI = {float(m1['ARI']):.12f}")
print(f"NMI = {float(m1['NMI']):.12f}")

print()
print("Run 2")
print(f"ARI = {float(m2['ARI']):.12f}")
print(f"NMI = {float(m2['NMI']):.12f}")

print()
print("Metric difference")
print("-----------------")
print(
    f"ARI difference = "
    f"{float(m2['ARI']) - float(m1['ARI']):+.12e}"
)
print(
    f"NMI difference = "
    f"{float(m2['NMI']) - float(m1['NMI']):+.12e}"
)

fields = [
    "total",
    "reconstruction",
    "sample_contrastive",
    "cluster_contrastive",
]

print()
print("Training trajectory")
print("-------------------")

for field in fields:
    a = np.array([float(row[field]) for row in h1])
    b = np.array([float(row[field]) for row in h2])

    diff = np.abs(a - b)

    idx = np.where(diff > 1e-10)[0]

    print()
    print(field)
    print(f"max abs diff = {diff.max():.12e}")

    if len(idx) == 0:
        print("first diff > 1e-10: NONE")
    else:
        first = int(idx[0])
        print(f"first diff > 1e-10: epoch {first + 1}")

print()
print("=" * 72)

SAME-SEED 50-EPOCH REPRODUCIBILITY CHECK

Run 1
ARI = 0.441822487702
NMI = 0.532442275377

Run 2
ARI = 0.440781107230
NMI = 0.531374671956

Metric difference
-----------------
ARI difference = -1.041380471598e-03
NMI difference = -1.067603421280e-03

Training trajectory
-------------------

total
max abs diff = 5.207061767578e-04
first diff > 1e-10: epoch 3

reconstruction
max abs diff = 4.969537258148e-06
first diff > 1e-10: epoch 7

sample_contrastive
max abs diff = 1.716613769531e-04
first diff > 1e-10: epoch 3

cluster_contrastive
max abs diff = 2.541542053223e-04
first diff > 1e-10: epoch 4



Cell 31：检查同一个 concat-Z embedding 上 KMeans 的稳定性

In [33]:
import numpy as np

from sklearn.cluster import KMeans
from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
)

Z = np.load(
    PROJECT_DIR / "results_clean/e185_clean_smoke/z_concat.npy"
)

GT = np.load(
    PROJECT_DIR / "results_clean/e185_clean_smoke/gt_labels.npy"
)

results = []

for kmeans_seed in range(10):
    pred = KMeans(
        n_clusters=14,
        n_init=20,
        random_state=kmeans_seed,
    ).fit_predict(Z)

    ari = adjusted_rand_score(GT, pred)

    nmi = normalized_mutual_info_score(
        GT,
        pred,
        average_method="max",
    )

    results.append((kmeans_seed, ari, nmi))

print("=" * 68)
print("KMEANS STABILITY ON THE SAME z_concat")
print("=" * 68)

for seed, ari, nmi in results:
    print(
        f"KMeans seed {seed:>2d} | "
        f"ARI = {ari:.6f} | "
        f"NMI = {nmi:.6f}"
    )

aris = np.array([x[1] for x in results])
nmis = np.array([x[2] for x in results])

print()
print("Summary")
print("-------")
print(f"ARI mean = {aris.mean():.6f}")
print(f"ARI std  = {aris.std():.6f}")
print(f"ARI min  = {aris.min():.6f}")
print(f"ARI max  = {aris.max():.6f}")

print()
print(f"NMI mean = {nmis.mean():.6f}")
print(f"NMI std  = {nmis.std():.6f}")
print(f"NMI min  = {nmis.min():.6f}")
print(f"NMI max  = {nmis.max():.6f}")

print("=" * 68)

KMEANS STABILITY ON THE SAME z_concat
KMeans seed  0 | ARI = 0.441822 | NMI = 0.532442
KMeans seed  1 | ARI = 0.440373 | NMI = 0.530524
KMeans seed  2 | ARI = 0.427741 | NMI = 0.520747
KMeans seed  3 | ARI = 0.404811 | NMI = 0.512764
KMeans seed  4 | ARI = 0.443522 | NMI = 0.536624
KMeans seed  5 | ARI = 0.435529 | NMI = 0.529599
KMeans seed  6 | ARI = 0.440878 | NMI = 0.530004
KMeans seed  7 | ARI = 0.433888 | NMI = 0.519545
KMeans seed  8 | ARI = 0.438056 | NMI = 0.523018
KMeans seed  9 | ARI = 0.420847 | NMI = 0.518667

Summary
-------
ARI mean = 0.432747
ARI std  = 0.011433
ARI min  = 0.404811
ARI max  = 0.443522

NMI mean = 0.525393
NMI std  = 0.007122
NMI min  = 0.512764
NMI max  = 0.536624


Cell 32：检查提高 KMeans n_init 后 concat-Z 聚类稳定性是否改善

In [34]:
import numpy as np

from sklearn.cluster import KMeans
from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
)

Z = np.load(
    PROJECT_DIR / "results_clean/e185_clean_smoke/z_concat.npy"
)

GT = np.load(
    PROJECT_DIR / "results_clean/e185_clean_smoke/gt_labels.npy"
)

n_init_values = [20, 50, 100]

print("=" * 78)
print("KMEANS STABILITY UNDER DIFFERENT n_init")
print("=" * 78)

for n_init in n_init_values:

    aris = []
    nmis = []

    for kmeans_seed in range(10):

        pred = KMeans(
            n_clusters=14,
            n_init=n_init,
            random_state=kmeans_seed,
        ).fit_predict(Z)

        ari = adjusted_rand_score(GT, pred)

        nmi = normalized_mutual_info_score(
            GT,
            pred,
            average_method="max",
        )

        aris.append(ari)
        nmis.append(nmi)

    aris = np.asarray(aris)
    nmis = np.asarray(nmis)

    print()
    print(f"n_init = {n_init}")
    print("-" * 30)

    print(
        f"ARI | mean={aris.mean():.6f} "
        f"std={aris.std():.6f} "
        f"min={aris.min():.6f} "
        f"max={aris.max():.6f}"
    )

    print(
        f"NMI | mean={nmis.mean():.6f} "
        f"std={nmis.std():.6f} "
        f"min={nmis.min():.6f} "
        f"max={nmis.max():.6f}"
    )

print()
print("=" * 78)

KMEANS STABILITY UNDER DIFFERENT n_init

n_init = 20
------------------------------
ARI | mean=0.432747 std=0.011433 min=0.404811 max=0.443522
NMI | mean=0.525393 std=0.007122 min=0.512764 max=0.536624

n_init = 50
------------------------------
ARI | mean=0.430502 std=0.012109 min=0.404811 max=0.443383
NMI | mean=0.525092 std=0.006237 min=0.512764 max=0.532442

n_init = 100
------------------------------
ARI | mean=0.423925 std=0.014222 min=0.396868 max=0.441822
NMI | mean=0.522700 std=0.007616 min=0.508843 max=0.532442



Cell 33：记录正式实验的 KMeans readout 协议

In [35]:
print("=" * 72)
print("FINAL KMEANS READOUT PROTOCOL")
print("=" * 72)

print("Official embedding : concat_z")
print("Clustering method  : KMeans")
print("n_init             : 20")
print("KMeans seed        : fixed to 0")
print("Training seed      : varied independently")
print()
print("Reason:")
print("- Increasing n_init from 20 to 50/100 did not reduce ARI variability.")
print("- KMeans initialization alone causes noticeable variation.")
print("- Fixing KMeans seed isolates training/model variability.")
print("- No KMeans seed is selected according to GT performance.")
print()
print("KMEANS PROTOCOL FROZEN")
print("=" * 72)

FINAL KMEANS READOUT PROTOCOL
Official embedding : concat_z
Clustering method  : KMeans
n_init             : 20
KMeans seed        : fixed to 0
Training seed      : varied independently

Reason:
- Increasing n_init from 20 to 50/100 did not reduce ARI variability.
- KMeans initialization alone causes noticeable variation.
- Fixing KMeans seed isolates training/model variability.
- No KMeans seed is selected according to GT performance.

KMEANS PROTOCOL FROZEN


50 轮 smoke 打包

In [36]:
import shutil
from pathlib import Path

SMOKE_DIR = PROJECT_DIR / "results_clean/e185_clean_smoke"

zip_path = shutil.make_archive(
    "/kaggle/working/e185_clean_smoke",
    "zip",
    root_dir=SMOKE_DIR,
)

print(zip_path)

/kaggle/working/e185_clean_smoke.zip
